# 03 - Modelagem

**Trabalho Final MBA BI Master - PUC-Rio**

Objetivo: treinar e comparar dois modelos de machine learning para previsao de
vendas - **Random Forest** e **XGBoost** - usando a base construida no notebook `02`.

Cada modelo e treinado em duas versoes: na escala original das vendas e na escala
logaritmica. Sao quatro treinos ao todo, e a comparacao entre as duas escalas e um
resultado do trabalho, nao apenas uma etapa tecnica (ver secao 2).

**Esquema de validacao:** divisao temporal unica, treino ate 18/06/2015 e teste nas
seis semanas seguintes. Nao se usa divisao aleatoria: embaralhar as linhas
significaria treinar com dias posteriores aos que se quer prever.

Validacao cruzada temporal com janelas deslizantes daria uma estimativa mais robusta,
mas multiplica o tempo de treino. Fica registrada como sugestao de trabalho futuro.

**Horizonte de previsao:** seis semanas a frente, mesmo desenho da competicao
Rossmann original.

## 0. Setup e carga

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import time
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

CAMINHO = '/content/drive/MyDrive/Trabalho MBA/'

base = pd.read_parquet(CAMINHO + 'base_modelagem_02.parquet')
base['Date'] = pd.to_datetime(base['Date'])

# tudo o que o notebook 02 deixou pronto, sem nada digitado a mao
with open(CAMINHO + 'features_02.txt') as f:
    features = f.read().strip().split('\n')

with open(CAMINHO + 'data_corte.txt') as f:
    DATA_CORTE = pd.Timestamp(f.read().strip())

baseline = pd.read_csv(CAMINHO + 'baseline_02.csv')

print('Base:', base.shape)
print('Features:', len(features))
print('Data de corte:', DATA_CORTE.date())
print()
print('Baseline do notebook 02:')
print(baseline.round(2).to_string(index=False))

Mounted at /content/drive
Base: (844338, 32)
Features: 30
Data de corte: 2015-06-19

Baseline do notebook 02:
                  modelo    rmse     mae  mape  rmspe
baseline_media_historica 1649.65 1240.35 18.27  23.32


## 1. Divisao treino/teste

O corte e o mesmo usado no notebook `02` para calcular as medias historicas. Precisa
ser identico: se fosse diferente, as medias teriam sido calculadas com dados que aqui
estariam no teste, e o vazamento voltaria pela porta dos fundos.

In [ ]:
treino = base[base['Date'] <  DATA_CORTE]
teste  = base[base['Date'] >= DATA_CORTE]

X_treino, y_treino = treino[features], treino['Sales']
X_teste,  y_teste  = teste[features],  teste['Sales']

print('Treino:', X_treino.shape, '|',
      treino['Date'].min().date(), 'a', treino['Date'].max().date())
print('Teste: ', X_teste.shape, '|',
      teste['Date'].min().date(), 'a', teste['Date'].max().date())

Treino: (802942, 30) | 2013-01-01 a 2015-06-18
Teste:  (41396, 30) | 2015-06-19 a 2015-07-31


### 1.1 Economia de memoria e retomada apos desconexao

Duas medidas para o Colab gratuito, que tem cerca de 12 GB de RAM e dois nucleos.

**`float32` no lugar de `float64`.** Metade da memoria para as colunas numericas,
sem perda relevante de precisao neste problema.

**Salvamento imediato.** Cada modelo e gravado no Drive assim que termina de treinar.
Se a sessao cair, basta reexecutar o notebook: os modelos ja treinados sao carregados
do disco em vez de treinados de novo. Para forcar um retreino, mude
`FORCAR_RETREINO` para `True` ou apague o arquivo `.pkl` correspondente.

In [ ]:
# float64 -> float32: metade da memoria
for col in X_treino.select_dtypes('float64').columns:
    X_treino[col] = X_treino[col].astype('float32')
    X_teste[col]  = X_teste[col].astype('float32')

print('Memoria do conjunto de treino:',
      f'{X_treino.memory_usage(deep=True).sum() / 1e6:.0f} MB')


FORCAR_RETREINO = False

def treinar_ou_carregar(arquivo, modelo, X, y_alvo):
    """Treina o modelo e salva no Drive. Se o arquivo ja existir, carrega
    em vez de treinar - permite retomar apos uma desconexao do Colab."""
    caminho = CAMINHO + arquivo

    if os.path.exists(caminho) and not FORCAR_RETREINO:
        print(f'{arquivo} ja existe - carregando do Drive (sem retreinar)')
        return joblib.load(caminho), 0.0

    inicio = time.time()
    modelo.fit(X, y_alvo)
    segundos = time.time() - inicio

    joblib.dump(modelo, caminho)
    print(f'{arquivo} treinado em {segundos/60:.1f} min e salvo')
    return modelo, segundos

Memoria do conjunto de treino: 151 MB


/tmp/ipykernel_739/2953099802.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_treino[col] = X_treino[col].astype('float32')
/tmp/ipykernel_739/2953099802.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_teste[col]  = X_teste[col].astype('float32')
/tmp/ipykernel_739/2953099802.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pand

## 2. Metricas e a questao da escala

### 2.1 As metricas

`RMSE` e `MAE` medem erro em euros. `MAPE` e `RMSPE` medem erro relativo, em
percentual. O **RMSPE** e a metrica oficial da competicao Rossmann, e e por ele
que a comparacao final se orienta.

### 2.2 Por que treinar tambem na escala logaritmica

XGBoost e Random Forest minimizam, por padrao, o **erro absoluto ao quadrado**. Uma
loja que vende 15.000 euros por dia contribui muito mais para essa soma do que uma
que vende 3.000, entao o modelo dedica sua capacidade as lojas grandes.

O RMSPE mede o oposto: erro **relativo**. Um erro de 500 euros vale 3,3% na loja
grande e 16,7% na pequena. Ou seja, o modelo e treinado para uma coisa e avaliado
por outra.

Treinar sobre `log(Sales)` alinha as duas. Minimizar erro absoluto no espaco
logaritmico equivale, aproximadamente, a minimizar erro percentual no espaco
original - porque a diferenca entre logaritmos e uma razao. A previsao e devolvida
a escala de euros com a operacao inversa antes de calcular qualquer metrica.

E uma tecnica conhecida das solucoes bem colocadas nessa competicao. O notebook `01`
ja havia observado que `log1p` deixa a distribuicao de vendas mais simetrica.

In [ ]:
def rmspe(y_real, y_previsto):
    """Root Mean Square Percentage Error - metrica oficial da competicao Rossmann.
    Dias com venda zero sao ignorados, conforme a regra do Kaggle."""
    y_real = np.asarray(y_real, dtype=float)
    y_previsto = np.asarray(y_previsto, dtype=float)

    mascara = y_real != 0
    erro_relativo = (y_real[mascara] - y_previsto[mascara]) / y_real[mascara]

    return np.sqrt(np.mean(erro_relativo ** 2)) * 100


def avaliar(nome, y_real, y_previsto, segundos=None):
    """Calcula as quatro metricas e devolve uma linha de resultado."""
    resultado = {
        'modelo': nome,
        'rmse':  np.sqrt(mean_squared_error(y_real, y_previsto)),
        'mae':   mean_absolute_error(y_real, y_previsto),
        'mape':  (abs(y_real - y_previsto) / y_real).mean() * 100,
        'rmspe': rmspe(y_real, y_previsto),
    }
    if segundos is not None:
        resultado['segundos'] = round(segundos, 1)

    print(f"{nome}")
    print(f"  RMSE:  {resultado['rmse']:,.0f}")
    print(f"  MAE:   {resultado['mae']:,.0f}")
    print(f"  MAPE:  {resultado['mape']:.2f}%")
    print(f"  RMSPE: {resultado['rmspe']:.2f}%")
    if segundos is not None:
        print(f"  tempo: {segundos:.1f}s")
    return resultado


resultados = []
previsoes = {}
print('Funcoes prontas.')

Funcoes prontas.


## 3. Random Forest

Random Forest e, de longe, a parte mais cara deste notebook. O custo nao vem do
numero de linhas em si, mas do **tamanho das arvores**, que cresce com elas.

Os parametros abaixo foram dimensionados para o Colab gratuito depois de uma primeira
versao que ultrapassou 15 minutos por treino e derrubou a sessao:

| Parametro | Efeito |
|---|---|
| `n_estimators=50` | metade das arvores |
| `min_samples_leaf=20` | folhas maiores, arvores muito menores - o de maior impacto |
| `max_samples=0.5` | cada arvore ve metade das linhas |
| `max_features=0.5` | cada divisao considera metade das variaveis |
| `n_jobs=2` | dois processos; `-1` acelera mas eleva o pico de memoria |

A perda de desempenho e pequena: Random Forest e robusto a esses parametros, e
`min_samples_leaf=20` inclusive reduz sobreajuste.

As duas versoes - escala original e logaritmica - usam **os mesmos parametros**, para
que a comparacao entre escalas seja justa.

In [ ]:
PARAMS_RF = dict(
    n_estimators=50,
    min_samples_leaf=20,    # nenhuma folha com menos de 20 registros
    max_samples=0.5,        # cada arvore ve metade das linhas
    max_features=0.5,       # cada divisao considera metade das variaveis
    n_jobs=2,               # -1 acelera, mas eleva o pico de memoria
    random_state=42,
)

rf, tempo_rf = treinar_ou_carregar('rf_original.pkl',
                                   RandomForestRegressor(**PARAMS_RF),
                                   X_treino, y_treino)

pred_rf = rf.predict(X_teste)
previsoes['Random Forest'] = pred_rf
resultados.append(avaliar('Random Forest (escala original)',
                          y_teste, pred_rf, tempo_rf))

rf_original.pkl ja existe - carregando do Drive (sem retreinar)
Random Forest (escala original)
  RMSE:  1,047
  MAE:   715
  MAPE:  10.66%
  RMSPE: 14.75%
  tempo: 0.0s


### 3.1 Random Forest na escala logaritmica

`log1p` calcula `log(1 + x)`, o que evita problema caso alguma venda seja zero.
`expm1` e a operacao inversa, aplicada a previsao para devolve-la a euros.

In [ ]:
rf_log, tempo_rf_log = treinar_ou_carregar('rf_log.pkl',
                                          RandomForestRegressor(**PARAMS_RF),
                                          X_treino, np.log1p(y_treino))

# a previsao sai em log; volta para euros antes de qualquer metrica
pred_rf_log = np.expm1(rf_log.predict(X_teste))
previsoes['Random Forest (log)'] = pred_rf_log
resultados.append(avaliar('Random Forest (escala log)',
                          y_teste, pred_rf_log, tempo_rf_log))

rf_log.pkl ja existe - carregando do Drive (sem retreinar)
Random Forest (escala log)
  RMSE:  1,043
  MAE:   708
  MAPE:  10.44%
  RMSPE: 14.37%
  tempo: 0.0s


## 4. XGBoost

`tree_method='hist'` agrupa os valores em faixas antes de procurar os pontos de
corte, o que torna o treino muito mais rapido em bases grandes sem perda relevante
de qualidade.

`subsample` e `colsample_bytree` fazem cada arvore ver apenas parte das linhas e das
colunas, o que reduz o sobreajuste.

In [ ]:
PARAMS_XGB = dict(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=8,
    subsample=0.8,          # cada arvore ve 80% das linhas
    colsample_bytree=0.8,   # cada arvore ve 80% das colunas
    tree_method='hist',
    n_jobs=-1,
    random_state=42,
)

xgb, tempo_xgb = treinar_ou_carregar('xgb_original.pkl',
                                    XGBRegressor(**PARAMS_XGB),
                                    X_treino, y_treino)

pred_xgb = xgb.predict(X_teste)
previsoes['XGBoost'] = pred_xgb
resultados.append(avaliar('XGBoost (escala original)',
                          y_teste, pred_xgb, tempo_xgb))

xgb_original.pkl ja existe - carregando do Drive (sem retreinar)
XGBoost (escala original)
  RMSE:  927
  MAE:   643
  MAPE:  9.78%
  RMSPE: 13.62%
  tempo: 0.0s


### 4.1 XGBoost na escala logaritmica

In [ ]:
xgb_log, tempo_xgb_log = treinar_ou_carregar('xgb_log.pkl',
                                            XGBRegressor(**PARAMS_XGB),
                                            X_treino, np.log1p(y_treino))

pred_xgb_log = np.expm1(xgb_log.predict(X_teste))
previsoes['XGBoost (log)'] = pred_xgb_log
resultados.append(avaliar('XGBoost (escala log)',
                          y_teste, pred_xgb_log, tempo_xgb_log))

xgb_log.pkl ja existe - carregando do Drive (sem retreinar)
XGBoost (escala log)
  RMSE:  893
  MAE:   611
  MAPE:  9.10%
  RMSPE: 12.51%
  tempo: 0.0s


## 5. Comparacao

Os quatro modelos frente ao baseline do notebook `02`. O ganho e calculado sobre o
RMSPE, metrica oficial da competicao.

A coluna `segundos` merece atencao na monografia. A diferenca de tempo entre os dois
algoritmos e de arquitetura, nao de ajuste: o `tree_method='hist'` do XGBoost agrupa
os valores em faixas antes de procurar os pontos de corte, em vez de testar cada valor
unico como faz o Random Forest. Eficiencia computacional e criterio real de escolha
entre modelos, sobretudo se a previsao for reprocessada com frequencia.

In [ ]:
comparacao = pd.DataFrame(resultados)

linha_baseline = baseline.iloc[0].to_dict()
linha_baseline['modelo'] = 'Baseline (media historica)'
comparacao = pd.concat([pd.DataFrame([linha_baseline]), comparacao],
                       ignore_index=True)

rmspe_baseline = float(baseline['rmspe'].iloc[0])
comparacao['ganho_rmspe_%'] = ((rmspe_baseline - comparacao['rmspe'])
                               / rmspe_baseline * 100).round(1)

colunas = ['modelo', 'rmse', 'mae', 'mape', 'rmspe', 'ganho_rmspe_%']
print(comparacao[colunas].round(2).to_string(index=False))

                         modelo    rmse     mae  mape  rmspe  ganho_rmspe_%
     Baseline (media historica) 1649.65 1240.35 18.27  23.32            0.0
Random Forest (escala original) 1046.93  715.34 10.66  14.75           36.8
     Random Forest (escala log) 1042.92  708.35 10.44  14.37           38.4
      XGBoost (escala original)  927.19  643.01  9.78  13.62           41.6
           XGBoost (escala log)  892.70  610.81  9.10  12.51           46.4


In [ ]:
melhor = comparacao.loc[comparacao['rmspe'].idxmin()]
print('Melhor modelo:', melhor['modelo'])
print(f"  RMSPE {melhor['rmspe']:.2f}% contra {rmspe_baseline:.2f}% do baseline")
print(f"  ganho de {melhor['ganho_rmspe_%']:.1f}%")

if melhor['rmspe'] >= rmspe_baseline:
    print('\\nATENCAO: nenhum modelo superou o baseline - revisar a modelagem')

Melhor modelo: XGBoost (escala log)
  RMSPE 12.51% contra 23.32% do baseline
  ganho de 46.4%


## 6. Importancia das variaveis

Responde a um dos objetivos declarados do trabalho: identificar quais indicadores
mais influenciam a previsao.

Uma ressalva de leitura. As tres medias historicas (`MediaLoja`,
`MediaLojaDiaSemana`, `MediaLojaPromo`) devem aparecer no topo, porque resumem o
patamar de vendas de cada loja - o principal fator de variacao da base. Isso e
esperado e nao indica problema. Para o objetivo do trabalho, o que interessa e a
ordenacao das **demais** variaveis entre si: promocao, calendario, concorrencia e
contexto macroeconomico. Vale apresentar as duas leituras separadamente na monografia.

In [ ]:
modelos = {'Random Forest': rf, 'Random Forest (log)': rf_log,
           'XGBoost': xgb, 'XGBoost (log)': xgb_log}

importancias = pd.DataFrame(
    {nome: modelo.feature_importances_ for nome, modelo in modelos.items()},
    index=features
)
importancias['media'] = importancias.mean(axis=1)
importancias = importancias.sort_values('media', ascending=False)

print('--- Todas as variaveis ---')
print((importancias['media'] * 100).round(2).to_string())

--- Todas as variaveis ---
MediaLojaPromo                45.91
MediaLojaDiaSemana            25.22
Promo                          8.14
MediaLoja                      3.97
DayOfWeek                      2.59
SemanaDoAno                    2.11
Dia                            1.82
DiaDoAno                       1.76
Desemprego_var12m              0.86
Mes                            0.78
InflacaoIndice_var12m          0.74
StateHoliday                   0.61
Trimestre                      0.56
ConfiancaConsumidor_dif12m     0.54
FimMes                         0.46
InicioMes                      0.43
StoreType                      0.39
VarejoVolume_var12m            0.39
Assortment                     0.38
LogDistConcorrente             0.33
SemDataConcorrente             0.26
Ano                            0.25
SchoolHoliday                  0.24
LojaComGap                     0.21
MesesConcorrente               0.21
MesesPromo2                    0.20
Store                          0.19
P

In [ ]:
# leitura sem as medias historicas, que dominam por construcao
medias_hist = ['MediaLoja', 'MediaLojaDiaSemana', 'MediaLojaPromo']
demais = importancias.drop(index=medias_hist)

# renormaliza para somar 100% entre as variaveis restantes
peso = demais['media'] / demais['media'].sum() * 100

print('--- Sem as medias historicas (peso relativo entre as demais) ---')
print(peso.round(2).to_string())

--- Sem as medias historicas (peso relativo entre as demais) ---
Promo                         32.71
DayOfWeek                     10.41
SemanaDoAno                    8.47
Dia                            7.30
DiaDoAno                       7.08
Desemprego_var12m              3.46
Mes                            3.15
InflacaoIndice_var12m          2.97
StateHoliday                   2.45
Trimestre                      2.25
ConfiancaConsumidor_dif12m     2.15
FimMes                         1.86
InicioMes                      1.72
StoreType                      1.57
VarejoVolume_var12m            1.56
Assortment                     1.54
LogDistConcorrente             1.34
SemDataConcorrente             1.03
Ano                            0.99
SchoolHoliday                  0.97
LojaComGap                     0.86
MesesConcorrente               0.86
MesesPromo2                    0.79
Store                          0.78
Promo2                         0.72
EmMesPromo2                    0.69

### 6.1 A variavel `Ano`

`Ano` assume apenas tres valores (2013, 2014, 2015) e o teste inteiro esta em 2015.
Se o modelo se apoiar nela para separar periodos, tera aprendido algo que nao se
sustenta fora da amostra - arvores nao extrapolam para um ano que nunca viram.

O teste abaixo retreina o melhor modelo sem `Ano` e compara.

**Antes de interpretar a diferenca, e preciso saber quanto o modelo varia sozinho.**
O XGBoost sorteia linhas (`subsample`) e colunas (`colsample_bytree`) a cada arvore,
entao dois treinos com sementes diferentes produzem resultados diferentes mesmo com
exatamente os mesmos dados. Essa variacao e o **ruido de fundo** do experimento.

Comparar "com Ano" contra "sem Ano" sem conhecer esse ruido leva a conclusoes falsas:
uma diferenca de meio ponto percentual pode ser efeito real da variavel ou apenas a
oscilacao natural entre dois treinos. A secao 6.2 mede o ruido primeiro; a 6.3 usa
essa medida como criterio.

### 6.2 Quanto o modelo varia sozinho

Treina o mesmo modelo, com os mesmos dados e as mesmas features, mudando apenas a
semente aleatoria. A variacao entre os resultados e o piso abaixo do qual nenhuma
diferenca pode ser considerada efeito de uma variavel.

In [ ]:
rmspe_sementes = []

for semente in [42, 7, 123]:
    params = dict(PARAMS_XGB)
    params['random_state'] = semente

    modelo_s, _ = treinar_ou_carregar(f'xgb_log_semente{semente}.pkl',
                                      XGBRegressor(**params),
                                      X_treino, np.log1p(y_treino))
    r = rmspe(y_teste, np.expm1(modelo_s.predict(X_teste)))
    rmspe_sementes.append(r)
    print(f'  semente {semente}: RMSPE {r:.2f}%')

RUIDO = max(rmspe_sementes) - min(rmspe_sementes)
print()
print(f'Amplitude entre sementes: {RUIDO:.2f} pontos percentuais')
print('Esse e o ruido de fundo: diferencas menores que isso nao sao conclusivas.')

xgb_log_semente42.pkl treinado em 1.3 min e salvo
  semente 42: RMSPE 12.51%
xgb_log_semente7.pkl treinado em 1.2 min e salvo
  semente 7: RMSPE 12.66%
xgb_log_semente123.pkl treinado em 1.3 min e salvo
  semente 123: RMSPE 12.66%

Amplitude entre sementes: 0.15 pontos percentuais
Esse e o ruido de fundo: diferencas menores que isso nao sao conclusivas.


### 6.3 Comparacao com e sem `Ano`

In [ ]:
features_sem_ano = [f for f in features if f != 'Ano']

xgb_sem_ano, tempo_sem_ano = treinar_ou_carregar(
    'xgb_log_sem_ano.pkl',
    XGBRegressor(**PARAMS_XGB),
    X_treino[features_sem_ano], np.log1p(y_treino))

pred_sem_ano = np.expm1(xgb_sem_ano.predict(X_teste[features_sem_ano]))
res_sem_ano = avaliar('XGBoost log, sem a variavel Ano',
                      y_teste, pred_sem_ano, tempo_sem_ano)

rmspe_com_ano = [r for r in resultados if r['modelo'] == 'XGBoost (escala log)'][0]['rmspe']
diferenca = res_sem_ano['rmspe'] - rmspe_com_ano

print()
print(f'RMSPE com Ano:  {rmspe_com_ano:.2f}%')
print(f'RMSPE sem Ano:  {res_sem_ano["rmspe"]:.2f}%')
print(f'Diferenca:      {diferenca:+.2f} pontos percentuais')
print(f'Ruido de fundo: {RUIDO:.2f} pontos percentuais (secao 6.2)')
print()

if abs(diferenca) <= RUIDO:
    print('A diferenca esta DENTRO do ruido entre sementes.')
    print('Nao ha evidencia de que Ano contribua; remove-la e defensavel')
    print('e simplifica a defesa do trabalho.')
else:
    print('A diferenca SUPERA o ruido entre sementes.')
    print('Ha evidencia de contribuicao real - manter Ano, declarando a limitacao:')
    print('a variavel tem apenas tres valores e nao se sustentaria para prever 2016.')

peso_ano = importancias.loc['Ano', 'media'] * 100
print()
print(f'Para contexto: Ano responde por {peso_ano:.2f}% da importancia media.')

xgb_log_sem_ano.pkl ja existe - carregando do Drive (sem retreinar)
XGBoost log, sem a variavel Ano
  RMSE:  921
  MAE:   635
  MAPE:  9.50%
  RMSPE: 13.06%
  tempo: 0.0s

RMSPE com Ano:  12.51%
RMSPE sem Ano:  13.06%
Diferenca:      +0.56 pontos percentuais
Ruido de fundo: 0.15 pontos percentuais (secao 6.2)

A diferenca SUPERA o ruido entre sementes.
Ha evidencia de contribuicao real - manter Ano, declarando a limitacao:
a variavel tem apenas tres valores e nao se sustentaria para prever 2016.

Para contexto: Ano responde por 0.25% da importancia media.


## 7. Conferencias do resultado

Diferente das conferencias do notebook `02`, que procuravam erro nos dados, estas
procuram erro de interpretacao do resultado. Sao cinco.

### 7.1 Previsoes negativas

A mais critica, e a unica capaz de invalidar as metricas. Modelos de regressao nao
sabem que venda nao pode ser negativa: nada nos dados impede a arvore de devolver
um valor abaixo de zero.

Na escala logaritmica o problema nao ocorre, porque `expm1` sempre devolve valor
positivo. Na escala original, pode ocorrer - e uma previsao negativa distorce todas
as metricas percentuais, ja que o erro relativo passa de 100%.

In [ ]:
for nome, pred in previsoes.items():
    negativos = (pred < 0).sum()
    marca = '  <- ATENCAO' if negativos else ''
    print(f'{nome}: {negativos} previsoes negativas | '
          f'minimo {pred.min():,.0f}{marca}')

Random Forest: 0 previsoes negativas | minimo 910
Random Forest (log): 0 previsoes negativas | minimo 887
XGBoost: 0 previsoes negativas | minimo 665
XGBoost (log): 0 previsoes negativas | minimo 767


### 7.2 Sobreajuste

Compara o erro no treino com o erro no teste. Alguma diferenca e normal e esperada -
o modelo sempre vai melhor onde foi treinado. O sinal de alarme e uma distancia
grande, que indicaria decoreba em vez de aprendizado.

In [ ]:
for nome, modelo, usa_log in [('Random Forest', rf, False),
                              ('Random Forest (log)', rf_log, True),
                              ('XGBoost', xgb, False),
                              ('XGBoost (log)', xgb_log, True)]:
    p_treino = modelo.predict(X_treino)
    p_teste  = modelo.predict(X_teste)
    if usa_log:
        p_treino, p_teste = np.expm1(p_treino), np.expm1(p_teste)

    r_treino = rmspe(y_treino, p_treino)
    r_teste  = rmspe(y_teste,  p_teste)
    print(f'{nome}')
    print(f'  RMSPE treino: {r_treino:.2f}% | teste: {r_teste:.2f}% '
          f'| distancia: {r_teste - r_treino:+.2f} pp')

Random Forest
  RMSPE treino: 19.04% | teste: 14.75% | distancia: -4.29 pp
Random Forest (log)
  RMSPE treino: 17.96% | teste: 14.37% | distancia: -3.60 pp
XGBoost
  RMSPE treino: 17.24% | teste: 13.62% | distancia: -3.62 pp
XGBoost (log)
  RMSPE treino: 13.30% | teste: 12.51% | distancia: -0.79 pp


### 7.3 Vies sistematico

Um modelo pode acertar na media geral e ainda assim errar de forma consistente para
cima ou para baixo. Em previsao de vendas isso importa: um vies de 3% significa
estoque sistematicamente dimensionado errado, mesmo com boa metrica de erro.

Atencao especial aos modelos em escala logaritmica. A transformacao inversa introduz
um **vies conhecido para baixo**, porque a media dos logaritmos nao corresponde ao
logaritmo da media (desigualdade de Jensen). Se ele aparecer aqui, e um achado real
para a monografia, nao um defeito da implementacao.

In [ ]:
media_real = y_teste.mean()

for nome, pred in previsoes.items():
    vies = (pred.mean() / media_real - 1) * 100
    print(f'{nome}: vies de {vies:+.2f}% '
          f'(previsto {pred.mean():,.0f} contra real {media_real:,.0f})')

Random Forest: vies de +0.82% (previsto 7,052 contra real 6,995)
Random Forest (log): vies de -0.16% (previsto 6,984 contra real 6,995)
XGBoost: vies de +1.71% (previsto 7,115 contra real 6,995)
XGBoost (log): vies de +0.47% (previsto 7,028 contra real 6,995)


### 7.4 Estabilidade ao longo do horizonte

O erro deve ser parecido na primeira e na sexta semana do teste. Se crescer ao longo
do periodo, o modelo degrada conforme a previsao se afasta do ultimo dado conhecido -
informacao direta sobre ate onde a previsao e confiavel, e que vale citar na monografia.

In [ ]:
melhor_pred = previsoes[[k for k in previsoes
                         if k.split(' (')[0] in nome_melhor_curto][0]]               if False else previsoes['XGBoost (log)']

erro_semanal = teste[['Date']].copy()
erro_semanal['erro_pct'] = abs(y_teste - melhor_pred) / y_teste * 100
erro_semanal['semana'] = (
    (erro_semanal['Date'] - erro_semanal['Date'].min()).dt.days // 7) + 1

print('Erro medio por semana do horizonte (XGBoost log):')
print(erro_semanal.groupby('semana')['erro_pct']
      .agg(['mean', 'count']).round(2).to_string())

Erro medio por semana do horizonte (XGBoost log):
         mean  count
semana              
1        8.14   6716
2       10.48   6721
3       10.82   6717
4        7.57   6710
5        8.69   6709
6        9.02   6710
7        8.40   1113


### 7.5 Erro por grupo

Onde o modelo vai pior. E o inicio da analise do notebook `04`, e a conferencia que
mais rende texto: uma diferenca sistematica entre grupos e um achado, nao um defeito.

In [ ]:
diagnostico = teste[['StoreType', 'LojaComGap', 'Promo']].copy()
diagnostico['erro_pct'] = abs(y_teste - melhor_pred) / y_teste * 100

for coluna in ['StoreType', 'LojaComGap', 'Promo']:
    print(f'Erro medio por {coluna}:')
    print(diagnostico.groupby(coluna)['erro_pct']
          .agg(['mean', 'count']).round(2).to_string())
    print()

Erro medio por StoreType:
           mean  count
StoreType             
0          9.15  22295
1          8.04    731
2          9.34   5476
3          8.98  12894

Erro medio por LojaComGap:
            mean  count
LojaComGap             
0           9.00  34713
1           9.62   6683

Erro medio por Promo:
       mean  count
Promo             
0      9.21  23581
1      8.95  17815



## 8. Exportacao

Os modelos e as previsoes alimentam o notebook `04` (analise de erro) e o dashboard.

In [ ]:
# tabela de previsoes: uma linha por loja/dia do teste, com o real e cada modelo
saida = teste[['Date', 'Store', 'Sales']].copy()
for nome, pred in previsoes.items():
    saida[nome] = pred

saida.to_parquet(CAMINHO + 'previsoes_03.parquet', index=False)
print('Previsoes salvas:', saida.shape)

comparacao.to_csv(CAMINHO + 'comparacao_modelos_03.csv', index=False)
print('Comparacao salva.')

importancias.to_csv(CAMINHO + 'importancia_variaveis_03.csv')
print('Importancias salvas.')

Previsoes salvas: (41396, 7)
Comparacao salva.
Importancias salvas.


In [ ]:
# so o melhor modelo vai para o dashboard - os arquivos sao grandes
nome_melhor = melhor['modelo']
mapa_objetos = {
    'Random Forest (escala original)': rf,
    'Random Forest (escala log)':      rf_log,
    'XGBoost (escala original)':       xgb,
    'XGBoost (escala log)':            xgb_log,
}

modelo_final = mapa_objetos[nome_melhor]
usa_log = 'log' in nome_melhor

joblib.dump({'modelo': modelo_final, 'features': features, 'escala_log': usa_log},
            CAMINHO + 'modelo_final_03.pkl')

print('Modelo salvo:', nome_melhor)
print('Escala logaritmica:', usa_log)
print('IMPORTANTE: para prever, aplicar np.expm1 na saida se escala_log for True')

Modelo salvo: XGBoost (escala log)
Escala logaritmica: True
IMPORTANTE: para prever, aplicar np.expm1 na saida se escala_log for True


---

**Proximo passo:** notebook `04 - Analise de resultados`, usando
`previsoes_03.parquet` e `importancia_variaveis_03.csv`.

**A declarar na monografia, independentemente da decisao sobre `Ano`:** a variavel
assume apenas tres valores e o conjunto de teste esta inteiramente em 2015. Modelos
de arvore nao extrapolam, entao o modelo treinado nao serviria para prever um ano
que nao viu, sem retreino. E uma limitacao do desenho, nao um defeito a corrigir.

**Pendencias para o 04:**

- Comparar o erro do modelo entre lojas com e sem `LojaComGap`. Se houver diferenca
  sistematica, avaliar uma variavel que marque os dias posteriores a reabertura.
- Analisar o erro por tipo de loja, por faixa de venda e por dia da semana.
- Verificar se `DiaDoAno` e `SemanaDoAno` disputam importancia por serem redundantes.